# Pipeline Harga Buah dari Hasil Scraping

Mengubah hasil scraping mentah (judul + harga) menjadi file referensi harga dengan skema yang sama seperti `price_reference_comprehensive.csv`, sehingga notebook awal bisa dipakai lagi tanpa banyak perubahan.

**Input** : `scraped_raw.csv` dengan kolom wajib: `date` (YYYY-MM-DD), `source`, `query_fruit`, `title`, `price_idr`

**Output**
- `price_reference_scraped.csv` (skema notebook awal + kolom `n_obs`, metode)
- `harga_harian.csv` (median harga/kg per buah per tanggal)

In [ ]:
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 1. Konfigurasi
Sesuaikan dengan 6 buah di model citramu.

In [ ]:
FRUITS = {
    "Apel":       [r"\bapel\b"],
    "Pisang":     [r"\bpisang\b"],
    "Jambu Biji": [r"\bjambu (biji|batu|kristal)\b"],
    "Jeruk":      [r"\bjeruk\b"],
    "Jeruk Nipis": [r"\bjeruk nipis\b", r"\blimau\b"],
    "Delima":     [r"\bdelima\b"],
}

# Produk olahan / non-buah yang sering ikut muncul di hasil pencarian marketplace.
EXCLUDE = [
    "bibit", "benih", "pohon", "stek", "jus", "juice", "keripik", "selai", "sirup",
    "syrup", "teh", "kopi", "parfum", "aroma", "rasa", "permen", "dodol", "manisan",
    "kering", "dried", "sale", "stik", "tepung", "pupuk", "pot", "cuka", "vinegar",
    "sabun", "lilin", "essential", "minyak", "kapsul", "ekstrak", "snack", "cuka",
    "kue", "cake", "es krim", "mainan", "plastik", "artificial", "hiasan",
]

# Kata kunci grade dari judul (klaim penjual, bukan penilaian mutu resmi!).
GOOD_WORDS = [r"\bsuper\b", r"\bpremium\b", r"\bgrade a\b", r"\bsortir\b",
              r"\bpilihan\b", r"\bjumbo\b", r"\bkualitas a\b", r"\bkw ?super\b"]
BAD_WORDS = [r"\bgrade b\b", r"\bgrade c\b", r"\bkw ?2\b", r"\breject\b",
             r"\bafkir\b", r"\bnon ?grade\b", r"\btanpa sortir\b", r"\bcampur\b",
             r"\bmemar\b", r"\bcacat\b"]

# Batas kewajaran harga per kg (membuang kesalahan parsing / harga iseng).
MIN_PER_KG, MAX_PER_KG = 3_000, 400_000
MIN_GROUP = 5          # minimal observasi agar grup grade berbasis kata kunci dipakai
DEFAULT_NOTE = "Local common"   # edit manual per buah: "Import premium", dst.

## 2. Fungsi parsing

In [ ]:
_NUM = r"(\d+(?:[.,]\d+)?)"


def parse_weight_kg(title: str):
    """Ambil berat (kg) dari judul. Kembalikan NaN jika tidak jelas."""
    t = str(title).lower()
    t = re.sub(r"(\d)\.(\d{3})\b", r"\1\2", t)  # "1.000 gr" -> "1000 gr"
    found = set()

    # pengali kemasan ("2 pack 500gr", "isi 6", "x2") -> total berat ambigu, buang
    if re.search(r"\b\d+\s*(pack|pak|pcs|pc|bungkus|box|dus|bks)\b|\bisi\s*\d+|\bx\s*\d+\b|\b\d+\s*x\b", t):
        return np.nan

    # pecahan dulu, lalu hapus dari teks agar "1/2 kg" tidak terbaca sebagai "2 kg"
    if re.search(r"setengah\s*(kg|kilo)|\b1/2\s*(kg|kilo)\b|\b0[.,]5\s*(kg|kilo)\b", t):
        found.add(0.5)
        t = re.sub(r"setengah\s*(kg|kilo)|\b1/2\s*(kg|kilo)\b", " ", t)
    for m in re.finditer(_NUM + r"\s*(kg|kilo(?:gram)?)\b", t):
        found.add(round(float(m.group(1).replace(",", ".")), 3))
    for m in re.finditer(_NUM + r"\s*(gr|gram|g)\b", t):
        found.add(round(float(m.group(1).replace(",", ".")) / 1000, 3))
    if not found and re.search(r"(per|/)\s*(kg|kilo)\b", t):
        found.add(1.0)

    found = {w for w in found if 0.05 <= w <= 25}
    return found.pop() if len(found) == 1 else np.nan   # ambigu -> buang


def match_fruit(title: str):
    t = str(title).lower()
    if any(re.search(rf"\b{re.escape(w)}\b", t) for w in EXCLUDE):
        return None
    hits = [f for f, pats in FRUITS.items() if any(re.search(p, t) for p in pats)]
    if "Jeruk Nipis" in hits:               # jeruk nipis != jeruk
        hits = ["Jeruk Nipis"]
    return hits[0] if len(hits) == 1 else None   # ambigu -> buang


def grade_flag(title: str):
    t = str(title).lower()
    good = any(re.search(p, t) for p in GOOD_WORDS)
    bad = any(re.search(p, t) for p in BAD_WORDS)
    if good and not bad:
        return "good"
    if bad and not good:
        return "bad"
    return "unk"

Uji cepat parser berat dan pencocokan buah:

In [ ]:
contoh = [
    "Apel Fuji Premium 1 kg",
    "Pisang Cavendish 500gr",
    "Jeruk Nipis Segar 1/2 kg",
    "Jambu Kristal 2 pack 500gr",
    "Jus Apel Asli 1 liter",
]
pd.DataFrame({
    "title": contoh,
    "berat_kg": [parse_weight_kg(x) for x in contoh],
    "buah": [match_fruit(x) for x in contoh],
    "grade": [grade_flag(x) for x in contoh],
})

## 3. Fungsi pipeline

In [ ]:
def clean(raw: pd.DataFrame):
    need = {"date", "source", "query_fruit", "title", "price_idr"}
    miss = need - set(raw.columns)
    if miss:
        raise ValueError(f"Kolom kurang: {sorted(miss)}")
    n0 = len(raw)
    df = raw.drop_duplicates(subset=["date", "source", "title", "price_idr"]).copy()
    n_dedup = len(df)

    df["fruit"] = df["title"].apply(match_fruit)
    df = df[df["fruit"].notna()]
    n_fruit = len(df)

    df["weight_kg"] = df["title"].apply(parse_weight_kg)
    df = df[df["weight_kg"].notna()]
    n_weight = len(df)

    df["price_idr"] = pd.to_numeric(df["price_idr"], errors="coerce")
    df["price_per_kg"] = df["price_idr"] / df["weight_kg"]
    df = df[df["price_per_kg"].between(MIN_PER_KG, MAX_PER_KG)]
    n_range = len(df)

    # outlier per buah: pagar 3xIQR pada skala log (lebar, supaya selisih grade
    # yang sah tidak ikut terbuang; hanya kesalahan ekstrem yang dibuang)
    def iqr_keep(s):
        ls = np.log(s)
        q1, q3 = ls.quantile([.25, .75])
        i = q3 - q1
        return ls.between(q1 - 3 * i, q3 + 3 * i)
    df = df[df.groupby("fruit")["price_per_kg"].transform(iqr_keep)]
    n_iqr = len(df)

    df["grade_flag"] = df["title"].apply(grade_flag)
    log = {"awal": n0, "setelah_dedup": n_dedup, "cocok_buah": n_fruit,
           "berat_terbaca": n_weight, "harga_wajar": n_range, "setelah_outlier": n_iqr}
    return df, log


def build_reference(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for fruit, g in df.groupby("fruit"):
        base = g["price_per_kg"].median()
        gg = g.loc[g.grade_flag == "good", "price_per_kg"]
        bb = g.loc[g.grade_flag == "bad", "price_per_kg"]
        good, gm = (gg.median(), "kata_kunci") if len(gg) >= MIN_GROUP else (g["price_per_kg"].quantile(.75), "kuantil_P75")
        bad, bm = (bb.median(), "kata_kunci") if len(bb) >= MIN_GROUP else (g["price_per_kg"].quantile(.25), "kuantil_P25")
        if not (bad < base < good):          # urutan tidak masuk akal -> pakai kuantil
            good, gm = g["price_per_kg"].quantile(.75), "kuantil_P75"
            bad, bm = g["price_per_kg"].quantile(.25), "kuantil_P25"
        rows.append({
            "Fruit": fruit,
            "BasePrice_IDR_kg": int(round(base)),
            "GoodQuality_IDR_kg": int(round(good)),
            "BadQuality_IDR_kg": int(round(bad)),
            "MarketIndo_IDR_kg": int(round(base)),   # semua harga sudah harga Indonesia
            "Form": "Fresh",
            "IndonesianMarketNote": DEFAULT_NOTE,
            "Source": "scraping:" + ",".join(sorted(g["source"].unique())),
            "n_obs": len(g), "n_good_kw": len(gg), "n_bad_kw": len(bb),
            "good_method": gm, "bad_method": bm,
            "date_min": g["date"].min(), "date_max": g["date"].max(),
        })
    return pd.DataFrame(rows)


def daily_series(df: pd.DataFrame) -> pd.DataFrame:
    return (df.groupby(["date", "fruit"])["price_per_kg"]
              .agg(median_per_kg="median", n="count").round(0).reset_index())

## 4. Jalankan
Ubah path di bawah sesuai lokasi file scraping kamu (di Colab: upload dulu lewat panel Files).

In [ ]:
RAW_CSV   = "scraped_raw.csv"
OUT_REF   = "price_reference_scraped.csv"
OUT_DAILY = "harga_harian.csv"

raw = pd.read_csv(RAW_CSV)
print(f"{len(raw):,} baris mentah")
raw.head()

In [ ]:
clean_df, log = clean(raw)
pd.Series(log, name="jumlah_baris").to_frame()

In [ ]:
ref = build_reference(clean_df)
ref.to_csv(OUT_REF, index=False)
daily = daily_series(clean_df)
daily.to_csv(OUT_DAILY, index=False)

ref[["Fruit", "BasePrice_IDR_kg", "GoodQuality_IDR_kg", "BadQuality_IDR_kg",
     "n_obs", "good_method", "bad_method"]]

In [ ]:
small = ref[ref.n_obs < 20]
if len(small):
    print("PERINGATAN: sampel < 20 untuk:", ", ".join(small.Fruit))
else:
    print("Semua buah punya sampel >= 20.")

## 5. Visualisasi (opsional)
Sebaran harga per kg tiap buah dan tren median harian.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
order = sorted(clean_df["fruit"].unique())
ax.boxplot([clean_df.loc[clean_df.fruit == f, "price_per_kg"] for f in order],
           labels=order, showfliers=False)
ax.set_ylabel("Harga (IDR/kg)")
ax.set_title("Sebaran harga per kg")
plt.tight_layout()
plt.show()

In [ ]:
piv = daily.pivot(index="date", columns="fruit", values="median_per_kg")
piv.index = pd.to_datetime(piv.index)
ax = piv.plot(figsize=(9, 4), marker="o")
ax.set_ylabel("Median harga (IDR/kg)")
ax.set_title("Median harga harian")
plt.tight_layout()
plt.show()